# nb04 — Micro Scale: ESP32, MCU, RPi Zero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb04_micro.ipynb)

> **The smallest wake-word models that still work — from 241 parameters to 50 K.**  
> Train every microcontroller-class tier in one pass, audit each one against a FLASH/RAM
> budget, benchmark its latency, and export a C header that drops straight into ESP-IDF or
> Arduino firmware. No GPU required — these models are tiny enough to train on a laptop CPU.

---

## What is "micro" scale?

A **wake word** (also called a *hotword* or *keyword*) is the short phrase that wakes a voice
assistant — "hey jarvis", "alexa", "ok google". A wake-word model listens to a continuous
audio stream and outputs a confidence score in `[0, 1]` for "did I just hear the phrase?".

On a phone or a Raspberry Pi this is easy. On a **microcontroller (MCU)** — an ESP32, an
ARM Cortex-M, a Raspberry Pi Zero — you have only kilobytes of memory and no floating-point
luxury. This notebook builds models small enough to live there.

Every micro tier here uses the same two-stage recipe:

- **MFCC featurizer** — *Mel-Frequency Cepstral Coefficients*. A fixed (non-learned) signal-
  processing front-end that turns 16 kHz raw audio into a compact spectral fingerprint. It
  has **zero trainable parameters**, so it costs no model size — only CPU time.
- **FFN head** — *Feed-Forward Network*, i.e. a plain two-layer fully-connected neural net
  (`Linear → ReLU → Linear → Sigmoid`). It takes the MFCC fingerprint and produces the
  confidence score. This is the only part with trainable weights, and the only part whose
  size we have to budget for.

There is no recurrence (no GRU/LSTM) and no attention — those are too heavy for an MCU. The
whole model is just two matrix multiplies.

---

## Target hardware and size budgets

| Tier | Extractor | Head | Params | ONNX (fp32) | int8 est. | Target |
|------|-----------|------|--------|-------------|-----------|--------|
| `esp32_nano` | MFCC-13 | FFN-16 | ~241 | <5 KB | <2 KB | ESP32, sub-1 KB weights |
| `esp32_sweet` | MFCC-13 | FFN-64 | ~1 K | ~10 KB | ~3 KB | ESP32, sub-10 KB |
| `esp32_max` | MFCC-13 | FFN-128 | ~2 K | ~15 KB | ~5 KB | ESP32, sub-50 KB |
| `micro` | MFCC-40 | FFN-128 | ~50 K | ~200 KB | ~50 KB | MCU / RPi Zero |
| `delta_micro` | delta-MFCC-13 | FFN-128 | ~55 K | ~220 KB | ~55 KB | MCU / RPi Zero |

- **MFCC-13 vs MFCC-40** — the number is how many cepstral coefficients are kept. 13 is the
  classic minimum (cheapest, smallest input layer); 40 captures more detail at a larger cost.
- **FFN-N** — `N` is the hidden-layer width. Bigger hidden = more capacity = more parameters.
- **delta-MFCC** — augments the 13 coefficients with their first and second time-derivatives
  ("deltas"), giving the model a sense of how the sound changes over time. Roughly the same
  size as plain `micro` but often a little more accurate.
- **int8** — *8-bit integer quantization*: storing each weight as one byte instead of a 4-byte
  float. This shrinks the model ~4× and lets MCUs without an FPU run it. The "int8 est." column
  is a rough `fp32 / 4` estimate.

**ESP32 constraints:**
- IRAM: 328 KB total, ~150 KB usable for code + weights
- PSRAM (WROVER variant): 4 MB — only this much headroom makes the `micro` tier feasible
- Without PSRAM, stick to `esp32_nano` or `esp32_sweet`

**RPi Zero 2W constraints:**
- 512 MB RAM — any tier fits comfortably
- Cortex-A53 × 4 — the `micro` tier runs at roughly 3 ms per frame

---

## How to read this notebook

The cells run top to bottom. You normally only edit **Cell 2 (Configuration)**, then
**Run All**. Here is what each step does:

| Cell | Step | What happens | Typical time (laptop CPU) |
|------|------|--------------|----------------------------|
| 2 | **Configure** | Set wake phrase, tiers, budgets | instant |
| 3 | **Install** | Install `ww_trainer` + audio deps | 3–5 min |
| 4 | **Dataset** | TTS positives + downloaded negatives | 10–20 min first run |
| 5 | **Train all tiers** | Loop over every micro tier, save F1 | 10–30 min |
| 6 | **Size audit** | Params + ONNX size vs ESP32 budget | instant |
| 7 | **Latency benchmark** | ms/inference + real-time factor | <1 min |
| 8 | **C header export** | `model.h` for the chosen ESP32 tier | instant |
| 9 | **Results table + plots** | Summary CSV and bar charts | <1 min |
| 10 | **Inference + CLI hints** | Sanity-check one positive sample | instant |

---

## Outputs

```
ww_output/
├── models/<tier>/model/best_f1.onnx           # classifier head (one per tier)
├── models/<tier>/model/best_f1_featurizer.onnx# MFCC front-end (one per tier)
├── models/<tier>/model/best_f1.pt             # PyTorch checkpoint (source for C export)
├── esp32_export/model.h                       # C header for the chosen ESP32 tier
├── micro_results.csv                          # full summary table
└── micro_results.png                          # F1 + size bar charts
```

---

## Glossary

- **F1** — the harmonic mean of precision and recall; a single 0–1 score for detector quality.
  Higher is better; ≥ 0.8 is usually deployable.
- **Latency** — wall-clock time for one inference call, in milliseconds.
- **RTF (Real-Time Factor)** — `latency / audio_duration`. RTF < 1 means faster than real time;
  for always-on listening you want it well under 0.1.
- **Quantization** — converting float weights to lower-precision integers (here, int8) to
  shrink the model and speed up MCU inference.


## Cell 2 — Configuration

**This is the only cell you normally edit.** Change `WAKE_WORD` to your phrase, optionally
tweak the budget knobs, then **Run All**.

Every value can also be supplied as an environment variable of the same name — handy for
running the notebook headless or on a server.

Key knobs:

- `WAKE_WORD` — the phrase to detect. Any language, any words.
- `TIER` — which **single** ESP32 tier gets exported as a C header in Cell 8 (training in
  Cell 5 always covers *all* micro tiers regardless of this).
- `MICRO_TIERS` — the full list of tiers trained and compared. Leave as-is to benchmark the
  whole micro family; trim it to train fewer and save time.
- `EPOCHS` — how many passes over the data. 20 is plenty for these tiny models.
- `N_POSITIVE` — number of synthetic wake-word clips to generate (300 is fine for micro).
- `SIZE_BUDGET_KB` — Cell 6 warns about any classifier head whose ONNX exceeds this.
- `SKIP_COMPLETED` — when `true`, a tier that already has a saved result JSON is skipped,
  so re-running the notebook resumes instead of retraining from scratch.
- `CUSTOM_TRAIN_CSV` / `CUSTOM_TEST_CSV` — "bring your own dataset" mode. Point these at your
  own labelled `path,label` CSVs to skip synthetic data generation entirely. If only a train
  CSV is given, it is auto-split 80/20.


In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Model ─────────────────────────────────────────────────────────────────────
# TIER controls which tier to export as C header at the end
TIER             = os.environ.get("TIER",             "esp32_sweet")
EPOCHS           = int(os.environ.get("EPOCHS",       "20"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "16"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "300"))
LANG             = os.environ.get("LANG",             "en")
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
REUSE_DATASET    = os.environ.get("REUSE_DATASET",    "true").lower() == "true"
CUSTOM_TRAIN_CSV = os.environ.get("CUSTOM_TRAIN_CSV", "")
CUSTOM_TEST_CSV  = os.environ.get("CUSTOM_TEST_CSV",  "")

# ── Micro-specific ────────────────────────────────────────────────────────────
# SIZE_BUDGET_KB: warn if any ONNX head file exceeds this
SIZE_BUDGET_KB   = int(os.environ.get("SIZE_BUDGET_KB", "50"))
SKIP_COMPLETED   = os.environ.get("SKIP_COMPLETED",   "true").lower() == "true"

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI       = os.environ.get("MLFLOW_URI",       "")
MLFLOW_SECRET    = os.environ.get("MLFLOW_SECRET",    "MLFLOW_TOKEN")

MICRO_TIERS = ["esp32_nano", "esp32_sweet", "esp32_max", "micro", "delta_micro"]
print(f"Wake word : {WAKE_WORD!r}")
print(f"Tiers     : {MICRO_TIERS}")
print(f"Epochs    : {EPOCHS}  |  Batch size: {BATCH_SIZE}")
print(f"C-export  : {TIER!r} (best ESP32 tier)")

## Cell 3 — Install dependencies and detect the platform

Installs `ww_trainer` together with the scientific stack (`torch`, `onnxruntime`, `pandas`,
`librosa`, …) and the data-generation plugins:

- `ovos-tts-plugin-edge-tts` — synthesises the positive (wake-word) audio.
- `vadonnx` — *Voice Activity Detection*, used to trim silence from clips.
- `datasets` — Hugging Face library for downloading negative speech and noise.

It then detects whether you are on Kaggle, Colab, Paperspace, or a local machine, and caps
the PyTorch thread count so training does not oversubscribe a shared CPU.

> **Expected runtime:** 3–5 min the first time (nothing cached); a few seconds on re-runs.
> If `ww_trainer` is already importable, the pip step that installs it is skipped.


In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "git+https://github.com/TigreGotico/vadonnx.git", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

print(f"Platform : {_platform}")
print(f"CPU cores: {os.cpu_count()}  |  torch threads: {torch.get_num_threads()}")
print(f"CUDA     : {torch.cuda.is_available()}")

## Cell 4 — Build (or reuse) the training dataset

A wake-word model learns from two kinds of labelled audio:

- **Positives** (label `1`) — recordings of your wake phrase. Here they are synthesised with
  edge-tts in many voices, speeds, and pitches so the model generalises across speakers.
- **Negatives** (label `0`) — audio that does *not* contain the phrase: ordinary speech and,
  when `ADVERSARIAL=true`, phonetically-similar confusables ("hay janice" for "hey jarvis")
  that teach the model to avoid near-miss false alarms.

When `DOWNLOAD_AUGMENT=true`, background noise, music, and *room impulse responses* (RIRs —
recordings that capture how a room reverberates) are also downloaded. They are mixed into the
training audio so the model stays robust in real, noisy rooms. Their folder paths are stashed
in `_aug_kwargs_full` and passed straight through to the trainer in Cell 5.

**Two modes:**

- **Synthetic (default):** generates everything from your `WAKE_WORD`. With
  `REUSE_DATASET=true`, if a dataset already exists under `OUTPUT_DIR/dataset/` it is reused
  instantly — no re-downloading on every run.
- **Bring-your-own:** set `CUSTOM_TRAIN_CSV` (and optionally `CUSTOM_TEST_CSV`) in Cell 2 to
  train on your own recordings instead.

A disk-space guard aborts early if less than 3 GB is free.

> **Expected runtime:** 10–20 min on the first synthetic run (mostly downloads); near-instant
> when reusing an existing dataset.

> **Troubleshooting:**
> - *`ModuleNotFoundError: datasets`* — re-run Cell 3.
> - *VAD/TTS plugin fails to load* — re-run Cell 3, or set `DOWNLOAD_AUGMENT=false` for a
>   lighter run.


In [ ]:
import shutil
from pathlib import Path

# Disk space guard
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — need at least 3 GB."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    if CUSTOM_TEST_CSV:
        test_csv = Path(CUSTOM_TEST_CSV)
    else:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED)
            random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for path, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(path, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
    print(f"BYO mode: train={train_csv}")
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        print(f"Reusing dataset at {dataset_dir}")
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
    else:
        print(f"Running datagen for '{WAKE_WORD}'...")
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD,
            output_dir=dataset_dir,
            n_positive=N_POSITIVE,
            lang=LANG,
            adversarial=ADVERSARIAL,
            vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT,
            seed=SEED,
        ))
    train_csv = _dr.train_csv
    test_csv  = _dr.test_csv
    if hasattr(_dr, "bg_noise_dir") and _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
        _aug_kwargs_full["bg_noise_folder"] = str(_dr.bg_noise_dir)
    if hasattr(_dr, "music_dir") and _dr.music_dir and Path(_dr.music_dir).exists():
        _aug_kwargs_full["music_folder"] = str(_dr.music_dir)
    if hasattr(_dr, "rir_dir") and _dr.rir_dir and Path(_dr.rir_dir).exists():
        _aug_kwargs_full["rir_folder"] = str(_dr.rir_dir)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

## Cell 5 — Train every micro tier

This is the heart of the notebook. It loops over **all** the tiers in `MICRO_TIERS` and, for
each one, calls `train_from_wakeword()` — the same one-shot training entry point used by the
quickstart. Each tier:

1. Reuses the dataset from Cell 4 (`reuse_dataset=True`).
2. Builds the tier's architecture (MFCC-X + FFN-Y, looked up from `tiers.py`).
3. Trains for `EPOCHS` epochs, keeping the checkpoint with the best **F1** on the test set.
4. Exports that best checkpoint to ONNX (a featurizer file and a head file).

Per-tier results (F1, precision, recall, elapsed time, ONNX paths) are written to
`OUTPUT_DIR/micro_results/<tier>.json` and collected in the `all_results` list that every
later cell consumes. Any tier that raises an exception is caught and recorded with a
`status` of `error:` rather than crashing the whole run, so one bad tier never blocks the rest.

With `SKIP_COMPLETED=true`, a tier whose JSON already exists is loaded from disk and skipped —
re-running the notebook resumes where it left off.

> **Expected runtime:** 10–30 min total on a laptop CPU for all five tiers, 20 epochs each.
> These models are tiny; most of the wall-clock time is feature extraction, not the network.

> **Troubleshooting:**
> - *F1 stays low (< 0.5) on `esp32_nano`* — expected for the absolute-minimum tier; it trades
>   accuracy for size. Compare against `esp32_sweet` / `micro` in the results table.
> - *All tiers error with a dataset path* — re-run Cell 4 so `train_csv` / `test_csv` exist.


In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

results_dir = Path(OUTPUT_DIR) / "micro_results"
results_dir.mkdir(parents=True, exist_ok=True)

all_results = []

for tier in MICRO_TIERS:
    result_file = results_dir / f"{tier}.json"
    model_subdir = Path(OUTPUT_DIR) / "models" / tier

    print(f"\n{'='*60}")
    print(f"Tier: {tier!r}")

    if SKIP_COMPLETED and result_file.exists():
        saved = json.loads(result_file.read_text())
        print(f"  SKIP (done): F1={saved.get('f1', 0):.4f}")
        all_results.append(saved)
        continue

    t0 = time.time()
    try:
        r = train_from_wakeword(
            WAKE_WORD,
            str(model_subdir),
            tier=tier,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
            reuse_dataset=True,
            **_aug_kwargs_full,
        )
        elapsed = time.time() - t0
        row = {
            "tier": tier,
            "f1": r.metrics.get("f1", 0.0),
            "precision": r.metrics.get("precision", 0.0),
            "recall": r.metrics.get("recall", 0.0),
            "elapsed_s": round(elapsed, 1),
            "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
            "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
            "status": "ok",
        }
        print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
    except Exception as exc:
        elapsed = time.time() - t0
        row = {
            "tier": tier, "f1": 0.0, "precision": 0.0, "recall": 0.0,
            "elapsed_s": round(elapsed, 1),
            "head_onnx": "", "feat_onnx": "",
            "status": f"error: {exc}",
        }
        print(f"  ERROR: {exc}")

    result_file.write_text(json.dumps(row, indent=2))
    all_results.append(row)

print(f"\nAll micro tiers done. {sum(1 for r in all_results if r['status']=='ok')}/{len(all_results)} succeeded.")

## Cell 6 — Size audit: does each tier fit its budget?

The whole point of micro scale is *fitting in tiny memory*. This cell builds a table that, per
tier, reports:

- **params** — approximate trainable-parameter count, read from the tier config.
- **head_onnx_kb** — actual on-disk size of the classifier head ONNX (the part you flash).
- **feat_onnx_kb** — size of the MFCC featurizer ONNX.
- **int8_est_kb** — a rough estimate of the head's size after int8 quantization (`fp32 / 4`).
- **within_budget** — whether the head fits its ESP32 flash budget. The check compares the
  fp32 ONNX size against `budget × 4`, since int8 quantization shrinks it about 4×.

`ESP32_BUDGETS` lists each tier's int8 KB target. After the table, any head exceeding the
`SIZE_BUDGET_KB` you set in Cell 2 is flagged.

> **Expected output:** a printed table plus a list of over-budget tiers (often empty). Runs
> instantly — it only stats files and reads metadata.


In [ ]:
import pandas as pd
from pathlib import Path

# ── Size audit ────────────────────────────────────────────────────────────────
# For each tier: parameter count, ONNX head size, estimated int8 size

ESP32_BUDGETS = {
    "esp32_nano": 1,   # KB
    "esp32_sweet": 10,
    "esp32_max": 50,
    "micro": 512,
    "delta_micro": 512,
}

audit_rows = []
for row in all_results:
    tier = row["tier"]
    head_path = Path(row["head_onnx"]) if row["head_onnx"] else None
    feat_path = Path(row["feat_onnx"]) if row["feat_onnx"] else None

    head_kb = head_path.stat().st_size / 1024 if head_path and head_path.exists() else None
    feat_kb = feat_path.stat().st_size / 1024 if feat_path and feat_path.exists() else None
    int8_est_kb = round(head_kb / 4, 1) if head_kb else None
    budget_kb = ESP32_BUDGETS.get(tier)
    fits = (head_kb is not None and budget_kb is not None and head_kb <= budget_kb * 4)

    # Try to get param count from tier info
    try:
        from ww_trainer.tiers import get_tier
        tier_cfg = get_tier(tier)
        params = getattr(tier_cfg, "approx_params", "?")
    except Exception:
        params = "?"

    audit_rows.append({
        "tier": tier,
        "params": params,
        "head_onnx_kb": round(head_kb, 1) if head_kb else "missing",
        "feat_onnx_kb": round(feat_kb, 1) if feat_kb else "missing",
        "int8_est_kb": int8_est_kb if int8_est_kb else "?",
        "budget_kb": budget_kb,
        "within_budget": "YES" if fits else ("NO" if head_kb else "N/A"),
        "f1": row.get("f1", 0.0),
        "status": row["status"],
    })

df_audit = pd.DataFrame(audit_rows)
print("Size audit:")
print(df_audit.to_string(index=False))
print()
print(f"SIZE_BUDGET_KB={SIZE_BUDGET_KB} KB — tiers exceeding this (head ONNX):")
for r in audit_rows:
    kb = r["head_onnx_kb"]
    if isinstance(kb, float) and kb > SIZE_BUDGET_KB:
        print(f"  {r['tier']}: {kb:.1f} KB  (budget: {SIZE_BUDGET_KB} KB)")

## Cell 7 — Latency benchmark

Size is half the story; **speed** is the other half. An always-on wake-word detector runs
continuously, so each inference must finish well inside the audio frame it is processing.

This cell times each tier's full ONNX pipeline (featurizer + head) on 1 second of random
audio and reports:

- **latency_ms** — mean wall-clock time for one inference call.
- **rtf** — *Real-Time Factor* = `latency / audio_duration`. With a 1-second clip,
  `rtf = latency_s`. RTF below 0.1 means the model keeps up with live audio with room to spare.

It prefers `ww_trainer.benchmark.measure_latency()` if available, and otherwise falls back to
timing `OnnxWakeWordInferencer.infer()` directly (with a short warm-up to discount one-time
session setup).

> **Note:** these numbers reflect *this machine's* CPU, not an ESP32. Treat them as relative
> rankings between tiers; on-device latency will differ. Runs in under a minute.


In [ ]:
import time
import numpy as np
from pathlib import Path

# ── Latency benchmark ─────────────────────────────────────────────────────────
# measure_latency() runs N forward passes on random 1-second audio and
# returns mean latency in milliseconds.

try:
    from ww_trainer.benchmark import measure_latency
    _have_benchmark = True
except ImportError:
    _have_benchmark = False
    print("ww_trainer.benchmark not available — using manual timing")

latency_rows = []
for row in all_results:
    if row["status"] != "ok":
        latency_rows.append({"tier": row["tier"], "latency_ms": None, "rtf": None})
        continue
    tier = row["tier"]
    feat_path = Path(row["feat_onnx"])
    head_path = Path(row["head_onnx"])
    if not feat_path.exists() or not head_path.exists():
        latency_rows.append({"tier": tier, "latency_ms": None, "rtf": None})
        continue

    if _have_benchmark:
        try:
            lat_ms = measure_latency(str(feat_path), str(head_path), n_runs=20)
        except Exception as e:
            lat_ms = None
            print(f"  {tier}: benchmark error: {e}")
    else:
        # Manual fallback: time OnnxWakeWordInferencer.infer()
        from ww_trainer.inference import OnnxWakeWordInferencer
        inf = OnnxWakeWordInferencer(str(feat_path), str(head_path))
        dummy = np.random.randn(16000).astype(np.float32)
        # Warm up
        for _ in range(3):
            inf.infer(dummy)
        t0 = time.perf_counter()
        for _ in range(20):
            inf.infer(dummy)
        lat_ms = (time.perf_counter() - t0) / 20 * 1000

    # RTF: latency / audio_duration (1 second of audio)
    rtf = (lat_ms / 1000) if lat_ms is not None else None
    print(f"  {tier}: {lat_ms:.1f} ms  (RTF={rtf:.4f})" if lat_ms else f"  {tier}: n/a")
    latency_rows.append({"tier": tier, "latency_ms": round(lat_ms, 2) if lat_ms else None, "rtf": round(rtf, 5) if rtf else None})

df_latency = pd.DataFrame(latency_rows)
print()
print(df_latency.to_string(index=False))

## Cell 8 — Export a C header for the firmware

This is the step that makes a model usable on an ESP32 *without* an ONNX runtime.
`export_to_c_header()` takes the trained FFN and writes a self-contained `model.h` containing:

- the int8-quantized weights as `const` C arrays (so they live in flash, not RAM),
- per-layer dequantization scale factors,
- a tiny `ww_model_infer(const float *input)` function — two matrix multiplies, a ReLU, and a
  sigmoid — with **no dependencies beyond standard C**. You `#include "model.h"` and call it.

It works only for **FFN heads**, which is exactly what every ESP32 tier uses (recurrent tiers
cannot be exported this way).

One subtlety: the exporter needs the live **PyTorch model**, not an ONNX file. So this cell
rebuilds the chosen tier's architecture from its config and loads the `best_f1.pt` checkpoint
that Cell 5 saved alongside the ONNX files, then hands that module to the exporter. It targets
the single tier named by `TIER` in Cell 2, falling back to another ESP32 tier if that one did
not train successfully.

> **Expected output:** a confirmation line, the embedded wake word, and a 30-line preview of
> `model.h`. Runs instantly.


In [ ]:
from pathlib import Path

# ── C header export for the best ESP32 tier ───────────────────────────────────
# export_to_c_header() writes a .h file with the int8-quantized FFN weights as
# const C arrays plus a tiny self-contained inference function (two matmuls +
# ReLU + sigmoid, no runtime dependencies). The header can be #include-d in an
# ESP-IDF or Arduino project.
#
# Important: export_to_c_header() takes a *PyTorch model* (an FfnClassifierHead),
# NOT an ONNX path. We therefore rebuild the model from its tier config, load the
# best_f1.pt checkpoint that training saved next to the head ONNX, and hand the
# live module to the exporter.

from ww_trainer.tiers import get_tier

esp32_export_dir = Path(OUTPUT_DIR) / "esp32_export"
esp32_export_dir.mkdir(parents=True, exist_ok=True)

# Find the result for the configured TIER
_c_export_row = next((r for r in all_results if r["tier"] == TIER and r["status"] == "ok"), None)
if _c_export_row is None:
    # Fall back to best available ESP32 tier
    for fallback in ["esp32_sweet", "esp32_nano", "esp32_max"]:
        _c_export_row = next((r for r in all_results if r["tier"] == fallback and r["status"] == "ok"), None)
        if _c_export_row:
            print(f"TIER={TIER!r} not available, falling back to {fallback!r}")
            TIER = fallback
            break


def _load_ffn_model(tier_name, ckpt_path):
    """Rebuild the tier's model and load its trained weights from best_f1.pt."""
    from ww_trainer.trainer import WakeWordTrainer
    tc = get_tier(tier_name)
    trainer = WakeWordTrainer(
        arch=tc.head_arch,
        featurizer="",
        feature_dim=None,
        featurizer_type=tc.extractor_type,
        wake_word=WAKE_WORD,
        device="cpu",
        hidden_dim=tc.hidden_dim,
        n_mfcc=tc.n_mfcc,
        losses_cfg=[{"name": "bce", "weight": 1.0}],
        export_onnx=False,
    )
    trainer.model.load_checkpoint(str(ckpt_path))
    trainer.model.eval()
    return trainer.model


if _c_export_row:
    head_onnx = Path(_c_export_row["head_onnx"])
    # best_f1.pt lives in the same directory as best_f1.onnx
    ckpt_path = head_onnx.with_name("best_f1.pt")
    c_header  = esp32_export_dir / "model.h"
    if not ckpt_path.exists():
        print(f"Checkpoint not found at {ckpt_path} — cannot export C header.")
    else:
        try:
            from ww_trainer.export_c import export_to_c_header
            _model = _load_ffn_model(TIER, ckpt_path)
            export_to_c_header(
                _model,
                c_header,
                model_name="ww_model",
                wake_word=WAKE_WORD,
            )
            size_kb = c_header.stat().st_size / 1024
            print(f"C header exported: {c_header}  ({size_kb:.1f} KB)")
            print(f"Wake word embedded in header: {WAKE_WORD!r}")
            print()
            # Print first 30 lines as preview
            lines = c_header.read_text().splitlines()[:30]
            print("--- model.h preview ---")
            for line in lines:
                print(line)
            print("...")
        except ImportError:
            print("ww_trainer.export_c not available — skipping C header export")
            print(f"Head ONNX for manual export: {head_onnx}")
        except TypeError as exc:
            # export_to_c_header only supports FFN heads (ESP32 tiers are all FFN);
            # a recurrent tier would land here.
            print(f"C export skipped — FFN-only exporter: {exc}")
else:
    print("No successful ESP32 tier found — cannot export C header.")


## Cell 9 — Results table and plots

Merges the training metrics, the size audit, and the latency benchmark into one DataFrame and
adds explicit `fits_esp32_nano / sweet / max` columns so you can see at a glance which tiers
clear which ESP32 budget. The combined table is saved to `OUTPUT_DIR/micro_results.csv`.

It then draws two bar charts — **F1 by tier** and **head ONNX size by tier** (with dashed
reference lines for the nano/sweet/max budgets) — and saves them to
`OUTPUT_DIR/micro_results.png`.

**How to read it:** look for the tier sitting in the sweet spot — high F1 *and* under your
flash budget. `esp32_sweet` is usually that tier; `esp32_nano` shows what the absolute floor
costs you in accuracy.

> **Expected output:** a printed comparison table, a saved CSV/PNG, and an inline figure.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Results table ─────────────────────────────────────────────────────────────

df_base = pd.DataFrame(all_results)
df_sz   = pd.DataFrame(audit_rows)
df_lat  = pd.DataFrame(latency_rows)

df = df_base.merge(df_sz[["tier","params","head_onnx_kb","int8_est_kb","within_budget"]],
                   on="tier", how="left")
df = df.merge(df_lat, on="tier", how="left")

# ESP32 fit flags
for esp_tier, budget_kb in [("esp32_nano", 5), ("esp32_sweet", 40), ("esp32_max", 60)]:
    col = f"fits_{esp_tier}"
    df[col] = df["head_onnx_kb"].apply(
        lambda kb: "YES" if isinstance(kb, (int, float)) and kb <= budget_kb else "NO"
    )

_cols = ["tier", "params", "head_onnx_kb", "int8_est_kb", "f1",
         "latency_ms", "fits_esp32_nano", "fits_esp32_sweet", "fits_esp32_max"]
_cols = [c for c in _cols if c in df.columns]

print("Micro tier results:")
print(df[_cols].to_string(index=False))

# Save CSV
csv_out = Path(OUTPUT_DIR) / "micro_results.csv"
df.to_csv(csv_out, index=False)
print(f"\nResults saved: {csv_out}")

# Bar chart
df_ok = df[df["status"] == "ok"].copy()
if not df_ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Micro tiers — {WAKE_WORD!r}", fontsize=12)

    ax = axes[0]
    ax.bar(df_ok["tier"], df_ok["f1"], color="steelblue", edgecolor="white")
    ax.set_ylabel("F1")
    ax.set_title("F1 by tier")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=30)

    ax2 = axes[1]
    if "head_onnx_kb" in df_ok.columns:
        sizes = pd.to_numeric(df_ok["head_onnx_kb"], errors="coerce")
        ax2.bar(df_ok["tier"], sizes, color="coral", edgecolor="white")
        ax2.set_ylabel("Head ONNX size (KB)")
        ax2.set_title("ONNX head size")
        ax2.tick_params(axis="x", rotation=30)
        for budget, label in [(1, "nano"), (10, "sweet"), (50, "max")]:
            ax2.axhline(budget, color="red", linestyle="--", linewidth=0.8, alpha=0.7)
            ax2.text(len(df_ok) - 0.5, budget + 0.5, f"ESP32 {label}", fontsize=7, color="red")

    plt.tight_layout()
    plot_path = str(Path(OUTPUT_DIR) / "micro_results.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot_path}")

## Cell 10 — Inference sanity check and next steps

Picks the highest-F1 successful tier, loads its ONNX pipeline with
`OnnxWakeWordInferencer` (the same class used at runtime in OpenVoiceOS — `onnxruntime` only,
no PyTorch), and scores one real positive sample from the test set. A score above `0.5` means
the detector fires on a true wake word, confirming the exported files actually work end to end.

It then prints ready-to-paste CLI commands for evaluating the model on your own WAV files and,
if a C header was produced, the `#include "model.h"` line for your firmware.

### Where to go next

| Goal | What to do |
|------|------------|
| Deploy the ONNX model on a Raspberry Pi / OVOS | See `notebooks/kaggle_quickstart.ipynb` (Cell 9 "Ship it") |
| Flash the C model onto an ESP32 | Use the generated `esp32_export/model.h`; see `docs/guides/embedded.md` |
| Squeeze even smaller (sub-1 KB) | Start from `esp32_nano`; see `examples/34_esp32_nano.py` |
| Search architectures automatically | Genetic search: `examples/35_esp32_genetic_search.py` |
| Train a bigger model for an RPi 3/4 | Use the embedded-scale notebook `notebooks/nb05_embedded.ipynb` |
| Penalise model size during training | `SizeAwareLoss` — see `docs/guides/embedded.md` |


In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── Inference test on a positive sample ───────────────────────────────────────
# Use the best-F1 successful tier for the inference demo.

best_row = sorted(
    [r for r in all_results if r["status"] == "ok"],
    key=lambda r: r.get("f1", 0),
    reverse=True,
)
best_row = best_row[0] if best_row else None

if best_row is None:
    print("No successful tier — cannot run inference test.")
else:
    feat_onnx = Path(best_row["feat_onnx"])
    head_onnx = Path(best_row["head_onnx"])

    if feat_onnx.exists() and head_onnx.exists():
        inferencer = OnnxWakeWordInferencer(str(feat_onnx), str(head_onnx))

        # Find a positive sample
        _pos_path = None
        with open(test_csv) as f:
            for row in csv.reader(f):
                if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                    _pos_path = row[0]
                    break

        if _pos_path:
            wav, sr = torchaudio.load(_pos_path)
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            wav_np = wav.mean(0).numpy().astype(np.float32)
            score = inferencer.infer(wav_np)
            print(f"Tier        : {best_row['tier']!r}  (F1={best_row['f1']:.4f})")
            print(f"Sample      : {Path(_pos_path).name}")
            print(f"Score       : {score:.4f}  ({'PASS ✓' if score > 0.5 else 'LOW — may need more training'})")
        else:
            print("No positive sample found in test CSV.")
    else:
        print(f"ONNX files missing for tier {best_row['tier']!r}")

print()
print("=" * 60)
print("CLI commands:")
if best_row and Path(best_row.get("feat_onnx", "")).exists():
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {best_row['feat_onnx']} \\")
    print(f"      --model      {best_row['head_onnx']} \\")
    print(f"      --audio      sample.wav")
c_header = Path(OUTPUT_DIR) / "esp32_export" / "model.h"
if c_header.exists():
    print(f"\nC header: {c_header}")
    print("Include in firmware:  #include \"model.h\"")
print("=" * 60)